In [ ]:
from pyfortmesa import mesa
import numpy as np
import matplotlib.pyplot as plt

mesa.set_cache_root(".")
mesa.set_inlist("inlist_eos_and_kap")

mix = mesa.composition({"h1": 0.70, "he4": 0.28, "c12": 0.02})

eos = mesa.Eos()
kap = mesa.Kap()


In [ ]:
T_eos = 1.0e7      # K
rho_eos = 1.0e2    # g/cm^3
T_kap = 1.0e6      # K
rho_kap = 1.0e-7   # g/cm^3

eos_out = eos.dt_full(T=T_eos, Rho=rho_eos, comp=mix)
kap_out = kap.opacity_full(T=T_kap, Rho=rho_kap, comp=mix)

print(eos_out["results"]["grad_ad"])
print(kap_out["kappa"])


In [ ]:
print(mesa.format_output_schema("eos_kap_profile"))
mesa.output_schema_names(include_aliases=False)


In [ ]:
# Fixed-composition eos profile on a base-10 log grid.
log_rho_array = np.linspace(-2.0, 8.0, 1000)

log_T_array = np.linspace(3.0, 8.0, 1000)


In [ ]:
eos_profile = eos.dt_profile(log_T_array, log_rho_array, mix.chem_id, mix.xa, input_mode="log10")

i_gamma1 = mesa.EOS_RESULT_NAMES.index("gamma1")
i_lnPgas = mesa.EOS_RESULT_NAMES.index("lnPgas")
gamma1 = eos_profile["results"][i_gamma1, :]
Pgas = np.exp(eos_profile["results"][i_lnPgas, :])
T_array = eos_profile["T"]
Prad = mesa.constants()["crad"] * T_array**4 / 3.0
Ptot = Pgas + Prad

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].plot(log_T_array, gamma1)
axes[0].set_xlabel(r"$\log_{10}(T/\mathrm{K})$")
axes[0].set_ylabel(r"$\Gamma_1$")

axes[1].semilogy(log_T_array, Pgas, label="Pgas")
axes[1].semilogy(log_T_array, Prad, label="Prad")
axes[1].semilogy(log_T_array, Ptot, label="Ptot", linestyle="--")
axes[1].set_xlabel(r"$\log_{10}(T/\mathrm{K})$")
axes[1].set_ylabel(r"pressure [dyn cm$^{-2}$]")
axes[1].legend()

plt.tight_layout()
plt.show()


In [ ]:
# kap profile call with caller-owned base-10 logT/logRho arrays.
log_T_kap = np.linspace(5.8, 6.4, 1000)
log_rho_kap = np.full_like(log_T_kap, -7.0)

kap_profile = kap.eos_kap_profile(log_T_kap, log_rho_kap, mix.chem_id, mix.xa, input_mode="log10")
opacity = kap_profile["kappa"]

plt.plot(log_T_kap, opacity)
plt.xlabel(r"$\log_{10}(T/\mathrm{K})$")
plt.ylabel(r"$\kappa$ [cm$^2$ g$^{-1}$]")
plt.show()


In [ ]:
mesa.shutdown()
